# 12. Псевдоразметка opXRD: отрицательный эксперимент

Этот этап сохранён как важная часть хронологии. Он показывает, почему 86 тысяч
неразмеченных opXRD нельзя было безопасно добавить в supervised FT.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


In [2]:
import subprocess

def run_script(script_name, *args):
    # Run one reproducible pipeline stage and stream its output.
    command = [sys.executable, str(PROJECT_ROOT / "src" / script_name), *map(str, args)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## Инференс и калибровка вероятностей

In [3]:
run_script("pseudo_labeling.py")

Running: d:\Dev\envs\study\python.exe D:\Users\user\Desktop\DS_XRD_project\src\pseudo_labeling.py


## Консервативные финальные пороги

In [4]:
run_script("pseudo_labeling_stage2.py")

Running: d:\Dev\envs\study\python.exe D:\Users\user\Desktop\DS_XRD_project\src\pseudo_labeling_stage2.py


## Итог эксперимента

In [5]:
import json
registry = json.loads((OUTPUTS_DIR / "pseudo_labels_registry.json").read_text(encoding="utf-8"))
display(registry)

{'created_utc': '2026-09-05T16:16:29.243509+00:00',
 'model': 'checkpoints/ft_v2_final.pt',
 'description': 'Псевдо-метки для неразмеченных opXRD-спектров (split_role=unlabeled в index_preprocessed.parquet). ЛАБОРАТОРНЫЕ метки (ft_pool_combined) не затронуты и остаются единственным источником ground truth.',
 'thresholds': {'crystal_system': {'threshold': 0.998,
   'basis': 'калибровка precision>=95% на ft_pool (оптимистично), консервативная надбавка'},
  'space_group': {'enabled': False,
   'reason': 'precision>=90% недостижим ни при каком пороге (калибровка на ft_pool)'},
  'elements': {'threshold': 0.85,
   'basis': 'поэлементная калибровка precision>=95% на ft_pool'},
  'lattice': {'enabled': False,
   'reason': 'регрессия без калибруемой уверенности - риск систематики'}},
 'counts': {'total_unlabeled': 86368,
  'labeled_any': 15951,
  'crystal_system': 16,
  'elements': 15950},
 'heads_note': 'строка может иметь псевдо-метку одной головы и не иметь другой; в FT это учитывается мас

## Вывод

SG и lattice не проходят надёжную калибровку; псевдометки не используются в последующих
моделях. Файл результата сохраняется только для воспроизводимости отрицательного опыта.